In [22]:
import pandas as pd 
import pycountry 

In [23]:
df = pd.read_csv("world-bank-income-groups/world-bank-income-groups.csv")

In [24]:
df.head()

,Entity,Code,Year,World Bank's income classification
0,Afghanistan,AFG,1987,Low-income countries
1,Afghanistan,AFG,1988,Low-income countries
2,Afghanistan,AFG,1989,Low-income countries
3,Afghanistan,AFG,1990,Low-income countries
4,Afghanistan,AFG,1991,Low-income countries


In [25]:
df["Code"][df["Year"] == 2025].nunique()

220

In [26]:
df.rename({"World Bank's income classification": "Income Group"}, axis=1, inplace=True)

In [27]:
# 1. Define the mapping dictionary
income_group_abbreviations = {
    'Low-income countries': 'LIC',
    'Lower-middle-income countries': 'LMIC',
    'Upper-middle-income countries': 'UMIC',
    'High-income countries': 'HIC'
}

# 2. Apply the mapping to your column
df['Income Group'] = df['Income Group'].map(income_group_abbreviations)

# 3. Add column to represent income type as number 
df['Income Type'] = df['Income Group'].map({'LIC': 1, 'LMIC': 2, 'UMIC': 3, 'HIC': 4})

In [28]:
income_groups_2025 = df[df["Year"] == 2025] 

In [29]:
def alpha3_to_alpha2(alpha3):
    """Convert 3-letter ISO country code to 2-letter. Returns None if not found."""
    try:
        return pycountry.countries.get(alpha_3=alpha3).alpha_2
    except AttributeError:
        return None

# Add 2-letter code column
income_groups_2025["Code_2"] = income_groups_2025["Code"].apply(alpha3_to_alpha2)

In [30]:
# Report anything that didn't convert
missing = income_groups_2025[income_groups_2025["Code_2"].isna()]
if len(missing) > 0:
    print(f"WARNING: {len(missing)} codes did not convert:")
    print(missing[["Entity", "Code"]].to_string())

               Entity      Code
1496  Channel Islands  OWID_CIS
3997           Kosovo  OWID_KOS


In [31]:
# Build lookup with 2-letter codes
income_group_mapping = dict(zip(
    income_groups_2025["Code_2"].dropna(),
    income_groups_2025.loc[income_groups_2025["Code_2"].notna(), "Income Type"]
))

In [32]:
print(f"Built lookup with {len(income_group_mapping)} countries")

Built lookup with 218 countries


In [33]:
df_a = pd.read_csv("cohortA_internal_with_doi.csv")
df_b = pd.read_csv("cohortB_internal_with_doi.csv")

cohort_countries = set(df_a["aff_country_code"].dropna()) | set(df_b["aff_country_code"].dropna())
lookup_countries = set(income_group_mapping.keys())
missing = cohort_countries - lookup_countries

if missing:
    print(f"Cohort countries NOT in lookup: {missing}")
    print("You'll need to add these manually before scoring will work.")
else:
    print(f"All {len(cohort_countries)} cohort countries covered by lookup ✓")

All 17 cohort countries covered by lookup ✓


In [36]:
import json
# Save lookup to JSON 
with open("income_group_lookup.json", "w") as f:
    json.dump(income_group_mapping, f, indent=4)